# Step2 評価品質レイヤ 実験ノートブック

epic [#22](https://github.com/shvalin1/speak-score/issues/22)（評価品質レイヤ `backend/src/services/*` 委譲）/ 子 [#21](https://github.com/shvalin1/speak-score/issues/21)（disfluency 精度改善）の **実験用**。
本番コード（`src/services/`）に移植する前に、実音声 + OpenAI で挙動を確かめる場。

- 方針: [ADR 004](../../docs/adr/004_verbatim_transcription_and_disfluency.md)（verbatim 文字起こし & disfluency）/ [ADR 005](../../docs/adr/005_evaluation_layer_ownership.md)（所有境界）/ [実装計画 002](../../docs/plans/002_step2_real_pipeline.md)（Phase 4/5）
- 既存スパイク `transcribe.py` / `audio_metrics.py` / `llm_eval.py` の関数を読み込んで対話的に回す。

## 使い方
1. カーネルは **`backend/.venv`** を選択（VSCode 右上のカーネル選択 → Python `backend/.venv`）。
2. `backend/.env` に `OPENAI_API_KEY=sk-...` を置く（または環境変数）。
3. 下の **AUDIO** に自分のサンプル音声/動画パスを設定（wav/m4a/mp3/mp4、≤25MB）。PII なのでコミット禁止。
4. 上から順に実行。


## 0. セットアップ — 共通 import / OpenAI クライアント / 音声パス

In [3]:
from pathlib import Path
import sys, json

def _find_eval_dir() -> Path:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        cand = base / "experiments" / "evaluation"
        if (cand / "transcribe.py").exists():
            return cand
        if base.name == "evaluation" and (base / "transcribe.py").exists():
            return base
    raise RuntimeError("experiments/evaluation が見つかりません（カーネルの作業ディレクトリを確認）")

EVAL_DIR = _find_eval_dir()
if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))
print("EVAL_DIR =", EVAL_DIR)

# 既存スパイクの関数を再利用（src には未だ入れない）
from transcribe import load_api_key, find_fillers, FILLER_PATTERNS
from audio_metrics import analyze, extract_to_wav, gen_sample_wav
from llm_eval import build_user_prompt, SYSTEM_PROMPT, JSON_SCHEMA, MODEL

from openai import OpenAI
key = load_api_key()
assert key, "OPENAI_API_KEY 未設定。backend/.env か環境変数に設定してください。"
client = OpenAI(api_key=key)
print("OpenAI client OK / model(llm) =", MODEL)

# ★ ここに自分のサンプル音声/動画パスを設定（PII のためコミット禁止）
AUDIO = "/Users/katohiroki/Documents/Labo/VScode/hakkutsu2026/speak-score/experiments/evaluation/InputData/2026-06-20-18.25.m4a"   # 例: "/Users/katohiroki/Desktop/kato_test.m4a"
assert AUDIO, "AUDIO にサンプル音声/動画のパスを設定してください"
assert Path(AUDIO).exists(), f"not found: {AUDIO}"
print("AUDIO =", AUDIO, f"({Path(AUDIO).stat().st_size/1024/1024:.2f} MB)")


EVAL_DIR = /Users/katohiroki/Documents/Labo/VScode/hakkutsu2026/speak-score/experiments/evaluation
OpenAI client OK / model(llm) = gpt-4o-2024-08-06
AUDIO = /Users/katohiroki/Documents/Labo/VScode/hakkutsu2026/speak-score/experiments/evaluation/InputData/2026-06-20-18.25.m4a (0.09 MB)


## 4a. 文字起こし（transcription） — verbatim 誘導 あり/なし 比較

ADR 004: Whisper は既定でフィラーを除去・正規化する。**フィラー保持 prompt + `temperature=0`** で verbatim を取得し source of truth にする。
誘導なし=フィラーがほぼ消える / 誘導あり=言い淀みが残る、を自分の音声で確認する。

In [4]:
VERBATIM_PROMPT = "えーと、あのー、まあ、なんか。フィラーや言い淀みも省略せず書き起こす。"

def transcribe_audio(path, prompt=None, model="whisper-1"):
    kwargs = dict(model=model, file=open(path, "rb"), language="ja", temperature=0)
    if prompt:
        kwargs["prompt"] = prompt
    if model == "whisper-1":
        kwargs["response_format"] = "verbose_json"
        kwargs["timestamp_granularities"] = ["segment"]
    return client.audio.transcriptions.create(**kwargs)

plain = transcribe_audio(AUDIO)                          # 誘導なし
verb  = transcribe_audio(AUDIO, prompt=VERBATIM_PROMPT)  # 誘導あり（verbatim）

print(f"[誘導なし] fillers={len(find_fillers(plain.text))}  chars={len(plain.text)}")
print(plain.text)
print("\n" + "=" * 60 + "\n")
print(f"[誘導あり] fillers={len(find_fillers(verb.text))}  chars={len(verb.text)}")
print(verb.text)

# 以降の 4b/4c では verbatim を真実として使う
transcript_text = verb.text


[誘導なし] fillers=0  chars=116
九州大学総合理工学部、総合理工学専攻の加藤裕樹です。 研究室ではエピデミオロジーのシミュレーションを行っています。 一年前の秋に上海交通大学に留学して、 そこではシミュレーションを専門で行っていました。 今日はよろしくお願いします。


[誘導あり] fillers=11  chars=175
えーと、九州大学総合理工学部、総合理工学専攻の加藤裕樹です。 研究室では、えーと、エピデミオロジーのシミュレーションを行っています。 うーん、一年前の秋に、えー、上海交通大学に留学して、 そこでは、えーと、シミュレーテッドアニーリング、漁師、漁師、えーと、漁師なんとかシミュレーションを、えーと、専門で行っておりました。 今日はよろしくお願いします。


In [5]:
# セグメント / no_speech_prob（whisper-1 verbose_json）。異常区間除去(Phase 4a)の感触を見る
for s in getattr(verb, "segments", None) or []:
    nsp = getattr(s, "no_speech_prob", None)
    flag = " ⚠no_speech" if (nsp is not None and nsp > 0.5) else ""
    nsp_s = f"{nsp:.2f}" if nsp is not None else "  - "
    print(f"[{s.start:6.2f}-{s.end:6.2f}] nsp={nsp_s}{flag}  {s.text}")


[  1.00-  8.00] nsp=0.12  えーと、九州大学総合理工学部、総合理工学専攻の加藤裕樹です。
[  9.00- 19.00] nsp=0.12  研究室では、えーと、エピデミオロジーのシミュレーションを行っています。
[ 20.00- 26.00] nsp=0.05  うーん、一年前の秋に、えー、上海交通大学に留学して、
[ 27.00- 40.00] nsp=0.05  そこでは、えーと、シミュレーテッドアニーリング、漁師、漁師、えーと、漁師なんとかシミュレーションを、えーと、専門で行っておりました。
[ 41.00- 43.00] nsp=0.05  今日はよろしくお願いします。


## 4b. 音声分析（audio_analysis / librosa）

`audio_metrics.analyze()` で話速 / 無音 / ピッチ / 音量を算出。`top_db`（無音判定閾値）は実データで要チューニング。
非wav/flac は ffmpeg で **WAV mono16k** に抽出してから解析（本番の入力フォーマットと同じ）。

In [6]:
# 非 wav/flac は WAV mono16k に抽出（plan 002: 入力は WAV mono16k 確定）
wav = AUDIO
if Path(AUDIO).suffix.lower() not in {".wav", ".flac"}:
    wav = extract_to_wav(AUDIO)   # ffmpeg -ac 1 -ar 16000 -vn
    print("ffmpeg extracted ->", wav)

# 話速の分子（句読点・空白を除いた文字数）は transcript 由来
chars = len(transcript_text.translate(str.maketrans("", "", "。、，．・ \u3000\n")))
metrics = analyze(wav, chars=chars, top_db=30)
print(json.dumps(metrics, ensure_ascii=False, indent=2))


ffmpeg extracted -> /var/folders/9g/nfgv1cwj6n7_lbnsr8fncm640000gn/T/tmpjrluc19t.wav


/Users/katohiroki/Documents/Labo/VScode/hakkutsu2026/speak-score/backend/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "params": {
    "top_db": 30,
    "chars": 151
  },
  "total_sec": 45.31,
  "voiced_sec": 45.02,
  "silence_ratio": 0.006,
  "silence_segments": [
    {
      "start": 0.0,
      "end": 0.29
    }
  ],
  "speech_rate_cpm": 201.2,
  "pitch_mean": 133.6,
  "pitch_std": 48.8,
  "volume_mean": 0.0696,
  "volume_cv": 0.638
}


In [7]:
# top_db スイープ: 自分の音声で無音判定がどう変わるか見て既定値を決める
for tdb in (20, 25, 30, 35, 40):
    m = analyze(wav, chars=chars, top_db=tdb)
    print(f"top_db={tdb:2d}  silence_ratio={m['silence_ratio']:.3f}  "
          f"voiced_sec={m['voiced_sec']:.2f}  silence_segs={len(m['silence_segments'])}  "
          f"speech_rate_cpm={m['speech_rate_cpm']}")


top_db=20  silence_ratio=0.078  voiced_sec=41.79  silence_segs=11  speech_rate_cpm=216.8
top_db=25  silence_ratio=0.008  voiced_sec=44.93  silence_segs=1  speech_rate_cpm=201.7
top_db=30  silence_ratio=0.006  voiced_sec=45.02  silence_segs=1  speech_rate_cpm=201.2
top_db=35  silence_ratio=0.005  voiced_sec=45.09  silence_segs=1  speech_rate_cpm=200.9
top_db=40  silence_ratio=0.004  voiced_sec=45.15  silence_segs=1  speech_rate_cpm=200.7


## 4c. LLM 評価（llm_evaluation / gpt-4o structured outputs）

content / structure の2軸採点 + strengths/improvements を JSON 強制（strict schema・`temperature=0`）。
delivery/confidence は `scoring.py` の決定論なので、metrics は **文脈として渡すが点数化はさせない**。

In [11]:
metrics_for_llm = {
    "speech_rate_cpm": metrics["speech_rate_cpm"],
    "filler_count": len(find_fillers(transcript_text)),
    "filler_rate": round(len(find_fillers(transcript_text)) / max(metrics["voiced_sec"] / 60, 1e-6), 2),
    "silence_ratio": metrics["silence_ratio"],
    "pitch_mean": metrics["pitch_mean"],
    "pitch_std": metrics["pitch_std"],
    "volume_cv": metrics["volume_cv"],
}
user_prompt = build_user_prompt(transcript_text, metrics_for_llm)
print("\n=== user_prompt ===\n", user_prompt)

resp = client.chat.completions.create(
    model=MODEL, temperature=0,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ],
    response_format={"type": "json_schema", "json_schema": JSON_SCHEMA},
)
result = json.loads(resp.choices[0].message.content)
print(json.dumps(result, ensure_ascii=False, indent=2))
print("\nusage:", resp.usage.prompt_tokens, "+", resp.usage.completion_tokens,
      "=", resp.usage.total_tokens, "tokens")



=== user_prompt ===
 # 応募者の回答（文字起こし）
えーと、九州大学総合理工学部、総合理工学専攻の加藤裕樹です。 研究室では、えーと、エピデミオロジーのシミュレーションを行っています。 うーん、一年前の秋に、えー、上海交通大学に留学して、 そこでは、えーと、シミュレーテッドアニーリング、漁師、漁師、えーと、漁師なんとかシミュレーションを、えーと、専門で行っておりました。 今日はよろしくお願いします。

# 音声メトリクス（参考・点数化しない）
{
  "speech_rate_cpm": 201.2,
  "filler_count": 11,
  "filler_rate": 14.66,
  "silence_ratio": 0.006,
  "pitch_mean": 133.6,
  "pitch_std": 48.8,
  "volume_cv": 0.638
}

上記を評価し、JSON スキーマに従って返してください。
{
  "content": {
    "score": 40,
    "comment": "応募者は自身の学歴と研究内容について述べているが、具体的な成果や数値がなく、説得力に欠ける。質問への適合性も不明確である。"
  },
  "structure": {
    "score": 30,
    "comment": "結論がなく、情報が断片的に提供されているため、論理の流れが不明瞭である。PREP法が活用されていない。"
  },
  "strengths": [
    "学歴と研究内容を簡潔に紹介している。",
    "留学経験をアピールしている。"
  ],
  "improvements": [
    "研究の具体的な成果や数値を示す。",
    "結論を先に述べ、論理的な流れを意識する。",
    "フィラーを減らし、話の流れをスムーズにする。"
  ]
}

usage: 621 + 195 = 816 tokens


## #21. disfluency 検出の実験（決定論ベースライン + 繰り返し検出）

issue #21 の推奨二段構え:
1. **決定論**: 形態素解析 + UniDic「感動詞-フィラー」（`fugashi` + `unidic-lite`）で語彙フィラーを抽出 → 再現性ある filler_rate。
2. **LLM**: 繰り返し/言い直し・機能語（その/あの/まあ/なんか）の曖昧性解消。

依存追加（Kato 所有＝image 再ビルド範囲）: `cd backend && uv add fugashi unidic-lite`

In [9]:
# (a) 語彙フィラー: 形態素ベースライン vs 単純パターン
try:
    import fugashi
    tagger = fugashi.Tagger()

    def morph_fillers(text):
        hits = []
        for w in tagger(text):
            f = w.feature
            pos1 = getattr(f, "pos1", None)
            pos2 = getattr(f, "pos2", None)
            if pos1 == "感動詞" and pos2 == "フィラー":
                hits.append(w.surface)
        return hits

    mf = morph_fillers(transcript_text)
    print(f"形態素(感動詞-フィラー): {len(mf)} 件 -> {mf}")
    print(f"単純パターン一致      : {len(find_fillers(transcript_text))} 件 "
          f"-> {[h['text'] for h in find_fillers(transcript_text)]}")
    print("\n→ 機能語(その/あの/まあ/なんか)を単純一致が誤検出していないか比較する")
except ModuleNotFoundError:
    print("fugashi 未導入。`cd backend && uv add fugashi unidic-lite` で追加してから再実行。")


fugashi 未導入。`cd backend && uv add fugashi unidic-lite` で追加してから再実行。


In [10]:
# (b) 繰り返し・言い直しの素朴な検出（連続同一トークンの塊）。LLM 化前のベースライン
import re

def repetitions(text):
    toks = re.findall(r"[一-龠ぁ-んァ-ヶー]+", text)
    reps, i = [], 0
    while i < len(toks):
        j = i
        while j + 1 < len(toks) and toks[j + 1] == toks[i]:
            j += 1
        if j > i:
            reps.append((toks[i], j - i + 1))
        i = j + 1
    return reps

print("連続反復:", repetitions(transcript_text) or "(なし)")


連続反復: [('漁師', 2)]


## まとめ / 次の一歩

- 4a/4b/4c の出力が妥当なら、関数を `backend/src/services/{transcription,audio_analysis,llm_evaluation}.py` に
  本番品質で移植（クライアント注入でモック可能に・エラー分類・per-call timeout）＋単体テスト追加。
- `top_db` / `filler_rate` 係数 / `scoring.py` の penalty 帯は、ここで見た実データ分布をもとに Phase 5 でチューニング。
- #21 の disfluency は (a) 形態素ベースライン → (b) LLM（削除のみ・捏造禁止のガードレール）で精緻化。